In [ ]:
import os 
import sys
import json

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

def is_project_root(path):
    path = Path(path)

    return (
        (path / "eytnet").is_dir()
        and (path / "configs").is_dir()
        and (path / "pyproject.toml").is_file()
    )

current = Path.cwd().resolve()

PROJECT_ROOT = next(
    (
        path for path in [current, *current.parents] if is_project_root(path)

    ),
    None,
)

if PROJECT_ROOT is None:
    content = Path("/content")
    if content.is_dir():
        PROJECT_ROOT = next(
            (
                path for path in content.iterdir() if ( path.is_dir() and is_project_root(path)
            )
        ),
        None,
    )

assert PROJECT_ROOT is not None, (
    "Proje bulunamadı. Önce Notebook 11'i "
    "aynı ortamda çalıştır."
)

os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from eytnet.metrics import (
    iou_matrix,
    load_ground_truth,
)

print("Proje:", PROJECT_ROOT)

In [ ]:
%pip install -q -U kagglehub

In [ ]:
import kagglehub
try:
    from kagglehub import drive
    drive.mount(
        "/content/drive",
        force_remount=False,
    )
except ImportError:
    print("Yerel ortam: drive bulunamadı")

def valid_dataset(path):
    path = Path(path)
    return all(
        (path / split / "images").is_dir()
        and (path / split / "labels").is_dir()
        for split in ["train", "val", "test"]
    )

local_data = PROJECT_ROOT / "data"

if valid_dataset(local_data):
    DATA_ROOT = local_data

else:
    downloaded_path =Path(
        kagglehub.dataset_download("pengbo00/home-fire-dataset")

    )

    candidates = [downloaded_path]

    for images_dir in downloaded_path.rglob(
        "images"
    ):
        if images_dir.parent.name == "train":
            candidates.append(
                images_dir.parent.parent
            )

    DATA_ROOT = next(
        path
        for path in candidates
        if valid_dataset(path)
    )

test_gt = load_ground_truth(DATA_ROOT,"test",)

TEST_IMAGE_DIR = (
    DATA_ROOT / "test" / "images"
)

print("Dataset:", DATA_ROOT)
print("Test görüntüsü:", len(test_gt))

In [ ]:
def find_result_directory(candidates):
    for path in candidates:
        path = path.resolve()

        if (
            (
                path
                / "test_predictions.npz"
            ).is_file()
            and
            (
                path
                / "test_results.json"
            ).is_file()
        ):
            return path

    return None


EYT_RUN_NAME = "eytnet_final_v1_opt_obj01"

EYT_RUN_DIR = find_result_directory(
    [
        (
            PROJECT_ROOT
            / "models"
            / "eytnet"
            / EYT_RUN_NAME
        ),
        (
            Path("/content/drive/MyDrive")
            / "EYTNet"
            / "runs"
            / EYT_RUN_NAME
        ),
    ]
)

FRCNN_RUN_NAME = (
    "fasterrcnn_v2_unfreeze_l3_"
    "step16_22_es_full"
)

FRCNN_RUN_DIR = find_result_directory(
    [
        (
            PROJECT_ROOT
            / "models"
            / "runs"
            / FRCNN_RUN_NAME
        ),
        (
            Path("/content/drive/MyDrive")
            / "early-fire-detection"
            / "models"
            / "runs"
            / FRCNN_RUN_NAME
        ),
    ]
)

assert EYT_RUN_DIR is not None, (
    "EYT-Net test sonuçları bulunamadı. "
    "Önce Notebook 11'i tamamla."
)

assert FRCNN_RUN_DIR is not None, (
    "Faster R-CNN test sonuçları bulunamadı. "
    "Yağız önce Notebook 08'i tamamlamalı."
)


def load_predictions(path):
    with np.load(path) as saved:
        return {
            key: saved[key]
            for key in saved.files
        }


def load_json(path):
    return json.loads(
        path.read_text(encoding="utf-8")
    )


eyt_predictions = load_predictions(
    EYT_RUN_DIR / "test_predictions.npz"
)

frcnn_predictions = load_predictions(
    FRCNN_RUN_DIR / "test_predictions.npz"
)

eyt_results = load_json(
    EYT_RUN_DIR / "test_results.json"
)

frcnn_results = load_json(
    FRCNN_RUN_DIR / "test_results.json"
)

drive_output = Path(
    "/content/drive/MyDrive/EYTNet/comparison"
)

if drive_output.parent.is_dir():
    OUTPUT_DIR = drive_output
else:
    OUTPUT_DIR = (
        PROJECT_ROOT
        / "models"
        / "comparison"
    )

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("EYT-Net:", EYT_RUN_DIR)
print("Faster R-CNN:", FRCNN_RUN_DIR)
print("Çıktılar:", OUTPUT_DIR)

In [ ]:
def comparison_row(name,result):
    overall = result["overall"]

    return {
        "yöntem": name,
        "mAP@0.5": overall["map50"],
        "mAP@0.5:0.95": overall["map5095"],
        "precision": overall["precision"],
        "recall": overall["recall"],
        "F1": overall["f1"],
        "F2": overall["f2"],
        "TP": overall["tp"],
        "FP": overall["fp"],
        "FN": overall["fn"],
        "ms/görüntü": result[
            "inference_ms_per_image"
        ],
        "FPS": result["fps"],
    }

comparison = pd.DataFrame(
    [
        comparison_row("EYT-Net", eyt_results),
        comparison_row("Faster R-CNN", frcnn_results),

    ]
)

display(comparison.round(4))

comparison.to_csv(
    OUTPUT_DIR / "model_comparison.csv",
    index=False,
)

metrics_to_plot = [
    "mAP@0.5",
    "mAP@0.5:0.95",
    "precision",
    "recall",
    "F1",
    "F2",
]

(
    comparison
    .set_index("yöntem")[metrics_to_plot]
    .T
    .plot(
        kind="bar",
        figsize=(11, 5),
    )
)

plt.ylabel("Değer")
plt.ylim(0,1)
plt.title("EYT-Net ve Faster R-CNN karşılaştırmasi")
plt.xticks(rotation=25)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()

plt.savefig(
    OUTPUT_DIR / "metric_comparison.png",
    dpi=160,
    bbox_inches="tight",
)

plt.show()


In [ ]:
def image_error_counts(
        predictions,
        ground_truth,
        score_threshold,
        iou_threshold=0.5,
):
    predictions = np.asarray(
        predictions,
        dtype = np.float32,  
    ).reshape(-1,6)

    ground_truth = np.asarray(
        ground_truth,
        dtype=np.float32,
    ).reshape(-1, 5)

    predictions = predictions[
        predictions[:, 4] >= score_threshold
    ]

    tp = 0
    fp = 0
    fn = 0

    for class_id in [0, 1]:
        class_predictions = predictions[
            predictions[:, 5].astype(int)
            == class_id
        ]

        class_ground_truth = ground_truth[
            ground_truth[:, 4].astype(int)
            == class_id
        ]

        if len(class_predictions):
            order = np.argsort(
                -class_predictions[:, 4]
            )

            class_predictions = (
                class_predictions[order]
            )

        used = np.zeros(
            len(class_ground_truth),
            dtype=bool,
        )

        for prediction in class_predictions:
            if len(class_ground_truth) == 0:
                fp += 1
                continue

            ious = iou_matrix(
                prediction[None, :4],
                class_ground_truth[:, :4],
            )[0]

            ious[used] = -1.0
            best_index = int(np.argmax(ious))

            if ious[best_index] >= iou_threshold:
                used[best_index] = True
                tp += 1
            else:
                fp += 1

        fn += int((~used).sum())

    return {
        "tp": tp,
        "fp": fp,
        "fn": fn,
    }

def build_error_table(
        predictions,
        score_threshold,
):
    rows = []

    for image_id, ground_truth in test_gt.items():
        current_predictions = predictions.get(
            image_id,
            np.zeros((0, 6), np.float32),
        )

        counts = image_error_counts(
            current_predictions,
            ground_truth,
            score_threshold,
        )

        rows.append(
            {
                "image_id": image_id,
                **counts,
            }
        )

    return pd.DataFrame(rows)

eyt_errors = build_error_table(
    eyt_predictions,
    eyt_results["score_threshold"],

)

frcnn_errors = build_error_table(
    frcnn_predictions,
    frcnn_results["score_threshold"],
)

error_summary = pd.DataFrame(
    [
        {
            "yöntem": "EYT-Net",
            "FP": int(eyt_errors["fp"].sum()),
            "FN": int(eyt_errors["fn"].sum()),
            "FP görüntüsü": int(
                (eyt_errors["fp"] > 0).sum()
            ),
            "FN görüntüsü": int(
                (eyt_errors["fn"] > 0).sum()
            ),
        },
        {
            "yöntem": "Faster R-CNN",
            "FP": int(
                frcnn_errors["fp"].sum()
            ),
            "FN": int(
                frcnn_errors["fn"].sum()
            ),
            "FP görüntüsü": int(
                (frcnn_errors["fp"] > 0).sum()
            ),
            "FN görüntüsü": int(
                (frcnn_errors["fn"] > 0).sum()
            ),
        },
    ]
)

display(error_summary)

eyt_errors.to_csv(
    OUTPUT_DIR / "eytnet_errors.csv",
    index=False,
)

frcnn_errors.to_csv(
    OUTPUT_DIR / "fasterrcnn_errors.csv",
    index=False,
)



In [ ]:
CLASS_NAMES = [
    "fire",
    "smoke",
]

CLASS_COLORS = {
    0:"red",
    1:"gray",
}

def show_error_examples(
    method_name,
    predictions,
    error_table,
    score_threshold,
    error_type,
    count=4,
):
    selected = (
        error_table
        .sort_values(
            error_type,
            ascending=False,
        )
        .query(f"{error_type} > 0")
        .head(count)
    )

    if selected.empty:
        print(
            f"{method_name} için {error_type} "
            "bulunamadı."
        )
        return

    fig, axes = plt.subplots(
        1,
        len(selected),
        figsize=(5 * len(selected), 5),
    )

    axes = np.atleast_1d(axes)

    for axis, row in zip(
        axes,
        selected.itertuples(),
    ):
        image_id = row.image_id

        image = Image.open(
            TEST_IMAGE_DIR
            / f"{image_id}.jpg"
        ).convert("RGB")

        axis.imshow(image)
        axis.axis("off")

        for (
            x1,
            y1,
            x2,
            y2,
            class_id,
        ) in test_gt[image_id]:
            axis.add_patch(
                plt.Rectangle(
                    (x1, y1),
                    x2 - x1,
                    y2 - y1,
                    fill=False,
                    edgecolor="lime",
                    linewidth=2,
                )
            )

            axis.text(
                x1,
                y1,
                (
                    "GT "
                    + CLASS_NAMES[
                        int(class_id)
                    ]
                ),
                color="black",
                fontsize=8,
                bbox={
                    "facecolor": "lime",
                    "alpha": 0.8,
                },
            )

        current_predictions = predictions.get(
            image_id,
            np.zeros((0, 6), np.float32),
        )

        current_predictions = (
            current_predictions[
                current_predictions[:, 4]
                >= score_threshold
            ]
        )

        for (
            x1,
            y1,
            x2,
            y2,
            score,
            class_id,
        ) in current_predictions:
            class_id = int(class_id)
            color = CLASS_COLORS[class_id]

            axis.add_patch(
                plt.Rectangle(
                    (x1, y1),
                    x2 - x1,
                    y2 - y1,
                    fill=False,
                    edgecolor=color,
                    linewidth=2,
                )
            )

            axis.text(
                x1,
                y2,
                (
                    f"PRED "
                    f"{CLASS_NAMES[class_id]} "
                    f"{score:.2f}"
                ),
                color="white",
                fontsize=8,
                bbox={
                    "facecolor": color,
                    "alpha": 0.8,
                },
            )

        axis.set_title(
            f"{image_id}\n"
            f"FP={row.fp}, FN={row.fn}"
        )

    plt.suptitle(
        f"{method_name} - "
        f"{error_type.upper()} örnekleri"
    )
    plt.tight_layout()
    plt.show()

In [ ]:
show_error_examples(
    "EYT-Net",
    eyt_predictions,
    eyt_errors,
    eyt_results["score_threshold"],
    eyt_type="fp",
)

show_error_examples(
    "EYT-Net",
    eyt_predictions,
    eyt_errors,
    eyt_results["score_threshold"],
    error_type="fn",
)

show_error_examples(
    "Faster R-CNN",
    frcnn_predictions,
    frcnn_errors,
    frcnn_results["score_threshold"],
    error_type="fp",
)

show_error_examples(
    "Faster R-CNN",
    frcnn_predictions,
    frcnn_errors,
    frcnn_results["score_threshold"],
    error_type="fn",
)